# Khám phá dữ liệu `crawl_v2` — chạy lại trên dữ liệu ĐÃ chuẩn hóa (bản sao của kham_pha_du_lieu.ipynb)

Notebook này **chỉ đọc và chỉ ra lỗi**, không sửa dữ liệu. Quy tắc sửa sẽ nằm trong `normalize.py`.

- Đầu vào: `data/crawl_v2_clean.json`
- Mỗi mục bên dưới kết thúc bằng một dòng **Kết luận** để ghi lại lỗi tìm được.

In [1]:
import json, re, unicodedata
from collections import Counter
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

DATA = Path("../data/crawl_v2_clean.json")
records = json.loads(DATA.read_text(encoding="utf-8"))
df = pd.DataFrame(records)
print(f"{len(df)} bản ghi, {len(df.columns)} cột")
df[["id", "category", "content_type", "title", "date"]].head(5)

141 bản ghi, 20 cột


,id,category,content_type,title,date
0,page-co-cau-to-chuc,chuc_nang_don_vi,page,Cơ cấu tổ chức,
1,unit-co-cau-to-chuc-phong-cong-nghe-thong-tin,chuc_nang_don_vi,unit,Phòng Công nghệ thông tin,
2,unit-co-cau-to-chuc-phong-cong-tac-chinh-tri-sinh-vien,chuc_nang_don_vi,unit,Phòng Công tác Chính trị sinh viên,
3,unit-co-cau-to-chuc-phong-dao-tao,chuc_nang_don_vi,unit,Phòng Đào tạo,
4,unit-co-cau-to-chuc-phong-hop-tac-quoc-te,chuc_nang_don_vi,unit,Phòng Hợp tác Quốc tế,


## 1. Tổng quan theo nhóm

In [2]:
print(df.groupby(["category", "content_type"]).size().unstack(fill_value=0))
print()
print("ID trùng:", df["id"].duplicated().sum())

content_type          announcement  major  page  unit
category                                             
chuc_nang_don_vi                 0      0     1    16
chuong_trinh_dao_tao             5     25     5     3
hoc_phi                         20      0     0     0
ke_hoach_dao_tao                10      0     0     0
lich_thi                        21      0     0     0
quy_che                         21      0     0     0
thong_tin_lien_lac               0      0     1     6
thong_tin_sinh_vien              0      0     2     0
tkb                              5      0     0     0

ID trùng: 0


Ghi chú:
- announcement: một thông báo hoặc quyết định của trường (ví dụ lịch thi, học phí), thường chỉ có tiêu đề kèm link file đính kèm.
- major: trang giới thiệu một ngành học, có mã ngành, học phí, cơ hội nghề nghiệp và danh sách môn học theo năm.
- page: một trang web lấy nguyên trang (ví dụ Cơ cấu tổ chức, Liên hệ, Đào tạo), thuộc loại trang chung của website.
- unit: một phòng hoặc khoa được tách ra từ trang page (ví dụ Phòng Đào tạo), có mô tả chức năng và thông tin liên hệ.


## 2. Độ dài nội dung — bản ghi chỉ có tiêu đề hoặc rỗng

In [3]:
df["n_chars"] = df["content"].str.len()
df["n_words"] = df["content"].str.split().str.len()
print(df.groupby("category")["n_chars"].describe()[["count", "min", "50%", "max"]].round(0))

# bản ghi có content gần như chỉ là tiêu đề (thông báo chỉ có file đính kèm)
def title_only(r):
    c = re.sub(r"\s+", " ", r["content"]).strip().lower()
    t = re.sub(r"\s+", " ", r["title"]).strip().lower()
    return c == t or len(c) < 30

df["title_only"] = df.apply(title_only, axis=1)
print()
print("Bản ghi content = tiêu đề / quá ngắn:", df["title_only"].sum(), "/", len(df))
print(df[df["title_only"]].groupby("category").size())

                      count    min     50%     max
category                                          
chuc_nang_don_vi       17.0  348.0   415.0   495.0
chuong_trinh_dao_tao   38.0   82.0  4004.0  7824.0
hoc_phi                20.0   60.0    96.0   240.0
ke_hoach_dao_tao       10.0   36.0    78.0   110.0
lich_thi               21.0   60.0    83.0   159.0
quy_che                21.0   62.0    87.0   852.0
thong_tin_lien_lac      7.0   91.0   103.0  1316.0
thong_tin_sinh_vien     2.0   16.0   772.0  1528.0
tkb                     5.0   63.0   109.0   174.0

Bản ghi content = tiêu đề / quá ngắn: 78 / 141
category
chuong_trinh_dao_tao     3
hoc_phi                 20
ke_hoach_dao_tao         9
lich_thi                21
quy_che                 19
thong_tin_sinh_vien      1
tkb                      5
dtype: int64


**Kết luận:** các thông báo `title_only` không có nội dung chữ thật, thông tin nằm trong file PDF ở `links`. Cần bước trích xuất PDF trước khi dùng cho RAG.

## 3. Unicode và ký tự lạ

In [4]:
def count_where(pred):
    return int(df["content"].map(pred).sum())

checks = {
    "không ở dạng NFC":            lambda s: unicodedata.normalize("NFC", s) != s,
    "có non-breaking space":       lambda s: "\u00a0" in s,
    "có zero-width / BOM":         lambda s: bool(re.search("[\u200b\u200c\u200d\ufeff]", s)),
    "có ký tự điều khiển":         lambda s: bool(re.search("[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", s)),
    "có ký tự thay thế (�)":       lambda s: "\ufffd" in s,
    "nháy/gạch kiểu Word (“ ” – —)": lambda s: bool(re.search("[\u201c\u201d\u2018\u2019\u2013\u2014]", s)),
    "có thẻ HTML":                 lambda s: bool(re.search(r"<[a-zA-Z/][^>]*>", s)),
    "có HTML entity (&nbsp; ...)": lambda s: bool(re.search(r"&[a-zA-Z]+;|&#\d+;", s)),
}
pd.Series({k: count_where(f) for k, f in checks.items()}, name="số bản ghi")

không ở dạng NFC                 0
có non-breaking space            0
có zero-width / BOM              0
có ký tự điều khiển              0
có ký tự thay thế (�)            0
nháy/gạch kiểu Word (“ ” – —)    0
có thẻ HTML                      0
có HTML entity (&nbsp; ...)      0
Name: số bản ghi, dtype: int64

## 4. Khoảng trắng, dòng trống và chuỗi rác

In [5]:
noise = {
    "dòng trống liên tiếp (3+ \\n)":     r"\n\s*\n\s*\n",
    "nhiều dấu cách liên tiếp":          r"[ \t]{2,}",
    "dòng chỉ có khoảng trắng":          r"(?m)^[ \t]+$",
    "dòng chỉ là '/'":                   r"(?m)^\s*/\s*$",
    "'Legacy URL'":                      r"Legacy URL",
    "'xem tại đây'":                     r"(?i)xem tại đây",
    "khoảng trắng đầu/cuối dòng":        r"(?m)(^[ \t]+\S|\S[ \t]+$)",
}
res = {k: int(df["content"].str.contains(p, regex=True).sum()) for k, p in noise.items()}
pd.Series(res, name="số bản ghi")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21488\2567976783.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  res = {k: int(df["content"].str.contains(p, regex=True).sum()) for k, p in noise.items()}


dòng trống liên tiếp (3+ \n)    0
nhiều dấu cách liên tiếp        0
dòng chỉ có khoảng trắng        0
dòng chỉ là '/'                 0
'Legacy URL'                    0
'xem tại đây'                   0
khoảng trắng đầu/cuối dòng      0
Name: số bản ghi, dtype: int64

In [6]:
# xem ví dụ thật của dòng rác
m = df[df["content"].str.contains(r"(?m)^\s*/\s*$", regex=True)]
for _, r in m.head(2).iterrows():
    print(r["id"], "|", repr(r["content"][:260]))

## 5. Chữ hoa/thường bất thường và biến thể chính tả

In [7]:
# từ có chữ HOA đứng ngay sau chữ thường, ví dụ 'ThÔNG' (không tính 'iPhone'-kiểu hợp lệ: chỉ xét từ tiếng Việt thuần)
def mixed_case(w):
    return any(w[i].isupper() and w[i - 1].islower() for i in range(1, len(w)))

words = Counter()
where = {}
for _, r in df.iterrows():
    for src in ("title", "content"):
        for w in re.findall(r"[^\W\d_]+", r[src]):
            if len(w) > 1 and mixed_case(w):
                words[w] += 1
                where.setdefault(w, r["id"])
print("Từ lẫn hoa/thường:", len(words), "dạng khác nhau")
print([(w, n, where[w]) for w, n in words.most_common(12)])

# tiêu đề viết hoa toàn bộ (THÔNG BÁO ...) so với viết thường
upper_titles = df["title"].map(lambda t: sum(c.isupper() for c in t) / max(1, sum(c.isalpha() for c in t)) > 0.6)
print("Tiêu đề phần lớn là chữ HOA:", int(upper_titles.sum()), "/", len(df))

variants = {
    "kỳ / kì":   (r"\bkỳ\b", r"\bkì\b"),
    "khóa / khoá": (r"\bkhóa\b", r"\bkhoá\b"),
    "hòa / hoà": (r"hòa", r"hoà"),
    "thủy / thuỷ": (r"thủy", r"thuỷ"),
}
alltext = "\n".join(df["title"] + "\n" + df["content"]).lower()
pd.DataFrame(
    {k: [len(re.findall(a, alltext)), len(re.findall(b, alltext))] for k, (a, b) in variants.items()},
    index=["dạng 1", "dạng 2"],
).T

Từ lẫn hoa/thường: 9 dạng khác nhau
[('iBT', 13, 'nganh-ke-toan'), ('YouTube', 1, 'unit-co-cau-to-chuc-phong-marketing-va-truyen-thong'), ('TikTok', 1, 'unit-co-cau-to-chuc-phong-marketing-va-truyen-thong'), ('PwC', 1, 'nganh-ke-toan-ACCA'), ('VietSun', 1, 'nganh-quan-tri-du-lich-lu-hanh'), ('GoEdu', 1, 'nganh-tai-chinh-ngan-hang'), ('JavaScript', 1, 'nganh-tri-tue-nhan-tao'), ('ThS', 1, 'page-dai-hoc-chinh-quy'), ('MobiFone', 1, 'page-dai-hoc-chinh-quy')]
Tiêu đề phần lớn là chữ HOA: 9 / 141


,dạng 1,dạng 2
kỳ / kì,348,0
khóa / khoá,88,0
hòa / hoà,3,2
thủy / thuỷ,0,0


**Kết luận:** cùng một từ được viết bằng hai kiểu đặt dấu (ví dụ `khóa` và `khoá`). Cần thống nhất, nếu không tìm kiếm từ khóa sẽ bỏ sót. Các từ lẫn hoa/thường còn lại (iBT, YouTube, PwC...) là tên riêng hợp lệ, **không sửa**.

## 6. Ngày tháng

In [8]:
ann = df[df["content_type"] == "announcement"]
print("Thông báo thiếu date:", (ann["date"] == "").sum(), "/", len(ann))
print("Định dạng date:", Counter(
    "YYYY-MM-DD" if re.fullmatch(r"\d{4}-\d{2}-\d{2}", d) else ("trống" if not d else "khác")
    for d in ann["date"]))
print("Khoảng ngày:", ann["date"].min(), "→", ann["date"].max())

# ngày dạng dd/mm/yyyy nằm trong text (cần chuẩn hóa hoặc giữ nguyên khi trích dẫn)
dmy = df["content"].str.count(r"\b\d{1,2}/\d{1,2}/\d{4}\b").sum()
dmy_t = df["title"].str.count(r"\b\d{1,2}/\d{1,2}/\d{4}\b").sum()
print("Ngày dạng d/m/yyyy trong content:", int(dmy), "| trong title:", int(dmy_t))

Thông báo thiếu date: 0 / 82
Định dạng date: Counter({'YYYY-MM-DD': 82})
Khoảng ngày: 2026-06-10 → 2026-10-26
Ngày dạng d/m/yyyy trong content: 14 | trong title: 4


## 7. Link

In [9]:
rows = []
for _, r in df.iterrows():
    for l in r["links"]:
        rows.append({"id": r["id"], "name": l["name"], "url": l["url"]})
lk = pd.DataFrame(rows)
print("Tổng link:", len(lk), "| bản ghi không có link:", int((df["links"].map(len) == 0).sum()))

def kind(u):
    if u.startswith("mailto:"): return "mailto"
    if u.startswith("tel:"):    return "tel"
    if "drive.google.com/file" in u: return "drive-file"
    if "drive.google.com/drive/folders" in u: return "drive-folder"
    if "docs.google.com" in u:  return "google-docs"
    if "media-cms" in u:        return "media-cms"
    if "thanglong.edu.vn" in u: return "trang-trường"
    return "khác"

lk["kind"] = lk["url"].map(kind)
print(lk["kind"].value_counts().to_string())
print()
print("Drive /view :", lk["url"].str.contains(r"drive.google.com/file/d/[^/]+/view").sum())
print("Drive /preview:", lk["url"].str.contains(r"drive.google.com/file/d/[^/]+/preview").sum())
print("Có ?usp=... :", lk["url"].str.contains(r"[?&]usp=").sum())
print("Còn localhost:", lk["url"].str.contains("localhost").sum())
print("Đuôi ảnh    :", lk["url"].str.contains(r"\.(png|jpe?g|gif|webp)(\?|$)", case=False).sum())

Tổng link: 140 | bản ghi không có link: 44
kind
drive-file      81
trang-trường    31
mailto          13
tel              5
drive-folder     3
khác             3
google-docs      3
media-cms        1

Drive /view : 1
Drive /preview: 80
Có ?usp=... : 7
Còn localhost: 0
Đuôi ảnh    : 0


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21488\158727714.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print("Đuôi ảnh    :", lk["url"].str.contains(r"\.(png|jpe?g|gif|webp)(\?|$)", case=False).sum())


**Kết luận:** link Drive có hai dạng (`/view`, `/preview`) và tham số thừa `?usp=`; cần quy về một dạng. Link ảnh cần OCR nếu chứa chữ.

## 8. Trùng lặp

In [10]:
dup_hash = df[df.duplicated("content_hash", keep=False)].sort_values("content_hash")
print("Bản ghi trùng content_hash:", len(dup_hash))
print(dup_hash[["id", "category", "title"]].head(10).to_string() if len(dup_hash) else "-")

norm_title = df["title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print()
print("Tiêu đề trùng nhau:", int(norm_title.duplicated().sum()))

Bản ghi trùng content_hash: 0
-

Tiêu đề trùng nhau: 5


## 9. Thiếu metadata

In [11]:
miss = pd.DataFrame({
    "thiếu department": df.groupby("content_type")["department"].apply(lambda s: int((s == "").sum())),
    "thiếu date":       df.groupby("content_type")["date"].apply(lambda s: int((s == "").sum())),
    "tổng":             df.groupby("content_type").size(),
})
miss

,thiếu department,thiếu date,tổng
content_type,,,
announcement,7,0,82
major,15,25,25
page,9,9,9
unit,0,25,25


**Kết luận:** tên khoa của ngành thiếu ở một số bản ghi; trang CMS (`page`) không có ngày.

## 10. Tổng hợp lỗi và quy tắc đề xuất cho `normalize.py`

In [12]:
summary = pd.DataFrame([
    ("Unicode chưa NFC, ký tự zero-width/BOM",   "unicodedata.normalize('NFC') + xóa ký tự vô hình"),
    ("Nháy, gạch kiểu Word (“ ” – —)",           "đổi về ký tự ASCII tương ứng"),
    ("Dòng trống thừa, dấu cách thừa, khoảng trắng đầu/cuối dòng", "gộp bằng regex"),
    ("Biến thể dấu: kỳ/kì, khóa/khoá, hòa/hoà",  "thống nhất một dạng (dùng dạng chiếm đa số)"),
    ("Ngày d/m/yyyy trong text",                 "giữ nguyên trong text, thêm bản ISO vào metadata"),
    ("Link Drive /view và /preview, tham số ?usp=", "quy về một dạng, bỏ tham số thừa"),
    ("Bản ghi chỉ có tiêu đề (đa số thông báo)", "đánh dấu cần trích xuất PDF, không cắt chunk riêng"),
    ("Tiêu đề trùng nhau",                       "kiểm tra thủ công, gộp hoặc giữ kèm ngày"),
    ("Thiếu department (ngành, trang CMS)",      "bổ sung từ nguồn khác hoặc để trống có chủ đích"),
], columns=["Lỗi", "Cách xử lý đề xuất"])
summary

,Lỗi,Cách xử lý đề xuất
0,"Unicode chưa NFC, ký tự zero-width/BOM",unicodedata.normalize('NFC') + xóa ký tự vô hình
1,"Nháy, gạch kiểu Word (“ ” – —)",đổi về ký tự ASCII tương ứng
2,"Dòng trống thừa, dấu cách thừa, khoảng trắng đầu/cuối dòng",gộp bằng regex
3,"Biến thể dấu: kỳ/kì, khóa/khoá, hòa/hoà",thống nhất một dạng (dùng dạng chiếm đa số)
4,Ngày d/m/yyyy trong text,"giữ nguyên trong text, thêm bản ISO vào metadata"
5,"Link Drive /view và /preview, tham số ?usp=","quy về một dạng, bỏ tham số thừa"
6,Bản ghi chỉ có tiêu đề (đa số thông báo),"đánh dấu cần trích xuất PDF, không cắt chunk riêng"
7,Tiêu đề trùng nhau,"kiểm tra thủ công, gộp hoặc giữ kèm ngày"
8,"Thiếu department (ngành, trang CMS)",bổ sung từ nguồn khác hoặc để trống có chủ đích


## Xem nội dung các chỗ còn lỗi

Mỗi kiểm tra bên dưới liệt kê **từng chỗ** còn khớp: bản ghi (bấm vào để mở trang nguồn), trường, và đoạn chữ xung quanh với phần lỗi được tô vàng. Khoảng trắng được hiện bằng `·` (dấu cách), `→` (tab), `⏎` (xuống dòng). Chỉ hiển thị 15 chỗ đầu của mỗi kiểm tra. Notebook chỉ đọc, **không sửa dữ liệu**; việc xử lý do bạn quyết định.

In [13]:
import html
from IPython.display import HTML, display

W = r"(?<![^\W\d_])"          # ranh giới từ cho chữ Unicode
E = r"(?![^\W\d_])"
CHECKS = {
    "Không ở dạng NFC":                 None,
    "Zero-width / BOM":                 r"[\u200b\u200c\u200d\u2060\ufeff\u00ad]",
    "Ký tự điều khiển":                 r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]",
    "Non-breaking space":               r"\u00a0",
    "Nháy / gạch kiểu Word (“ ” ‘ ’ – —)": r"[\u201c\u201d\u2018\u2019\u2013\u2014]",
    "Dòng trống liên tiếp (3+ \\n)":     r"\n\s*\n\s*\n",
    "Nhiều dấu cách liên tiếp":          r"[ \t]{2,}",
    "Dòng chỉ có khoảng trắng":         r"(?m)^[ \t]+$",
    "Khoảng trắng đầu/cuối dòng":       r"(?m)(^[ \t]+\S|\S[ \t]+$)",
    "'Legacy URL'":                     r"Legacy URL",
    "'xem tại đây'":                    r"(?i)xem tại đây",
    "Biến thể: kì (thay vì kỳ)":        W + r"kì" + E,
    "Biến thể: khoá (thay vì khóa)":    W + r"khoá" + E,
    "Biến thể: hoà (thay vì hòa)":      W + r"hoà" + E,
    "Biến thể: thuỷ (thay vì thủy)":    W + r"thuỷ" + E,
}
SHOW, CTX = 15, 45

def vis(s):
    return html.escape(s).replace(" ", "·").replace("\t", "→").replace("\n", "⏎")

def find_hits(name, pat):
    hits = []
    for _, r in df.iterrows():
        for f in ("title", "content"):
            text = r[f]
            if pat is None:
                if unicodedata.normalize("NFC", text) != text:
                    bad = next(i for i, (a, b) in enumerate(zip(text, unicodedata.normalize("NFC", text))) if a != b)
                    hits.append((r, f, text, bad, bad + 1))
                continue
            for m in re.finditer(pat, text):
                hits.append((r, f, text, m.start(), m.end()))
    return hits

total = 0
for name, pat in CHECKS.items():
    hits = find_hits(name, pat)
    total += len(hits)
    n_rec = len({h[0]["id"] for h in hits})
    display(HTML(f"<h4>{html.escape(name)}: {len(hits)} chỗ / {n_rec} bản ghi</h4>"))
    if not hits:
        continue
    rows = []
    for r, f, text, s, e in hits[:SHOW]:
        a, b = max(0, s - CTX), min(len(text), e + CTX)
        snippet = vis(text[a:s]) + "<mark>" + vis(text[s:e]) + "</mark>" + vis(text[e:b])
        link = r.get("source_url") or ""
        line_no = text.count(chr(10), 0, s) + 1
        idc = f'<a href="{html.escape(link)}" target="_blank">{html.escape(r["id"])}</a><br><small>{html.escape(link)}</small>' if link else html.escape(r["id"])
        atts = r.get("attachments")
        atts = atts if isinstance(atts, list) else []
        if atts:
            idc += "<br><b>File đính kèm:</b>" + "".join(
                f'<br>• <a href="{html.escape(a.get("url", ""))}" target="_blank">{html.escape(a.get("name", "file"))}</a>'
                for a in atts)
        rows.append(f"<tr><td>{idc}</td><td>{f}<br><small>dòng {line_no}</small></td><td style='font-family:monospace;white-space:pre-wrap'>…{snippet}…</td></tr>")
    more = f"<p>… còn {len(hits) - SHOW} chỗ nữa.</p>" if len(hits) > SHOW else ""
    display(HTML("<table><tr><th>Bản ghi (bấm hoặc copy địa chỉ)</th><th>Trường / vị trí</th><th>Nội dung</th></tr>" + "".join(rows) + "</table>" + more))
print("Tổng số chỗ còn khớp:", total)


Tổng số chỗ còn khớp: 0
